In [2]:
import pandas as pd
df = pd.read_csv("data/processed/movies_cleaned.csv")

print(df.shape)
df.head()

(9967, 12)


,adult,id,original_language,original_title,overview,popularity,release_date,title,vote_average,vote_count,movie_id,embedding_text
0,False,278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,46.3708,1994-09-23,The Shawshank Redemption,8.718,30171,278,Imprisoned in the 1940s for the double murder ...
1,False,238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",42.0006,1972-03-14,The Godfather,8.686,22787,238,"Spanning the years 1945 to 1955, a chronicle o..."
2,False,240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.8671,1974-12-20,The Godfather Part II,8.571,13812,240,In the continuing saga of the Corleone crime f...
3,False,424,en,Schindler's List,The true story of how businessman Oskar Schind...,24.2944,1993-12-15,Schindler's List,8.567,17341,424,The true story of how businessman Oskar Schind...
4,False,389,en,12 Angry Men,The defense and the prosecution have rested an...,19.4971,1957-04-10,12 Angry Men,8.559,9908,389,The defense and the prosecution have rested an...


In [4]:
from transformers import AutoTokenizer
import pandas as pd

c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
models = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
    "gte_modernbert": "Alibaba-NLP/gte-modernbert-base",
    #"embeddinggemma": "google/embeddinggemma-300m", kasnije
}

In [6]:
token_length_results = []

texts = df["embedding_text"].tolist()

for model_name, model_id in models.items():
    print(f"Obrada: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    token_lengths = [
        len(
            tokenizer(
                text,
                add_special_tokens=True,
                truncation=False,
            )["input_ids"]
        )
        for text in texts
    ]

    token_length_results.append(
        {
            "model": model_name,
            "min_tokens": min(token_lengths),
            "mean_tokens": sum(token_lengths) / len(token_lengths),
            "median_tokens": pd.Series(token_lengths).median(),
            "p90_tokens": pd.Series(token_lengths).quantile(0.90),
            "p95_tokens": pd.Series(token_lengths).quantile(0.95),
            "p99_tokens": pd.Series(token_lengths).quantile(0.99),
            "max_tokens": max(token_lengths),
            "over_256_count": sum(length > 256 for length in token_lengths),
            "over_256_percentage": (
                sum(length > 256 for length in token_lengths)
                / len(token_lengths)
                * 100
            ),
        }
    )

Obrada: minilm


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Obrada: mpnet


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Obrada: gte_modernbert


c:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\IMDB_WEEK1_SIX\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikola.bakic\.cache\huggingface\hub\models--Alibaba-NLP--gte-modernbert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [7]:
token_length_summary = pd.DataFrame(token_length_results)

display(token_length_summary)

,model,min_tokens,mean_tokens,median_tokens,p90_tokens,p95_tokens,p99_tokens,max_tokens,over_256_count,over_256_percentage
0,minilm,6,58.162837,52.0,96.0,108.0,145.00,224,0,0.0
1,mpnet,6,58.162837,52.0,96.0,108.0,145.00,224,0,0.0
2,gte_modernbert,6,59.296478,53.0,97.0,110.0,146.34,235,0,0.0


In [8]:
import gc

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

In [9]:
model_registry = {
    "minilm": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "expected_dimension": 384,
    },
    "mpnet": {
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "expected_dimension": 768,
    },
    "gte_modernbert": {
        "model_id": "Alibaba-NLP/gte-modernbert-base",
        "expected_dimension": 768,
    },
}

smoke_test_texts = df["embedding_text"].head(8).tolist()
smoke_test_results = []

for model_name, config in model_registry.items():
    print(f"Testiranje modela: {model_name}")

    model = SentenceTransformer(
        config["model_id"],
        device="cpu",
    )
    model.max_seq_length = 256

    embeddings = model.encode(
        smoke_test_texts,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    smoke_test_results.append(
        {
            "model": model_name,
            "shape": embeddings.shape,
            "expected_dimension": config["expected_dimension"],
            "contains_nan": bool(np.isnan(embeddings).any()),
            "contains_inf": bool(np.isinf(embeddings).any()),
            "mean_vector_norm": float(
                np.linalg.norm(embeddings, axis=1).mean()
            ),
            "passed": (
                embeddings.shape
                == (len(smoke_test_texts), config["expected_dimension"])
                and not np.isnan(embeddings).any()
                and not np.isinf(embeddings).any()
            ),
        }
    )

    del model, embeddings
    gc.collect()

Testiranje modela: minilm


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5638.90it/s]


Testiranje modela: mpnet


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4557.26it/s]


Testiranje modela: gte_modernbert


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 4570.56it/s]


In [10]:
smoke_test_summary = pd.DataFrame(smoke_test_results)
display(smoke_test_summary)

,model,shape,expected_dimension,contains_nan,contains_inf,mean_vector_norm,passed
0,minilm,"(8, 384)",384,False,False,1.0,True
1,mpnet,"(8, 768)",768,False,False,1.0,True
2,gte_modernbert,"(8, 768)",768,False,False,1.0,True


In [11]:
from pathlib import Path
from huggingface_hub.constants import HF_HUB_CACHE

for path in Path(HF_HUB_CACHE).glob("models--*"):
    print(path.name)

models--Alibaba-NLP--gte-modernbert-base
models--sentence-transformers--all-MiniLM-L6-v2
models--sentence-transformers--all-mpnet-base-v2


In [12]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [13]:
embeddings_directory = Path("artifacts/embeddings")
metrics_directory = Path("artifacts/metrics")

embeddings_directory.mkdir(parents=True, exist_ok=True)
metrics_directory.mkdir(parents=True, exist_ok=True)

texts = df["embedding_text"].tolist()

benchmark_results = []

In [14]:
for model_name, config in model_registry.items():
    print(f"\nModel: {model_name}")

    model = SentenceTransformer(
        config["model_id"],
        device="cpu",
    )
    model.max_seq_length = 256

    # Warm-up se ne računa u glavno vreme.
    model.encode(
        texts[: min(32, len(texts))],
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    start_time = time.perf_counter()

    embeddings = model.encode(
        texts,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    encode_time = time.perf_counter() - start_time

    output_path = embeddings_directory / f"{model_name}_embeddings.npy"
    np.save(output_path, embeddings)

    benchmark_results.append(
        {
            "model": model_name,
            "number_of_movies": len(texts),
            "embedding_dimension": embeddings.shape[1],
            "encode_time_seconds": encode_time,
            "movies_per_second": len(texts) / encode_time,
            "embedding_file_mb": output_path.stat().st_size / (1024 ** 2),
        }
    )

    print(f"Shape: {embeddings.shape}")
    print(f"Vreme: {encode_time:.2f} s")
    print(f"Brzina: {len(texts) / encode_time:.2f} filmova/s")
    print(f"Sačuvano: {output_path}")

    del model, embeddings
    gc.collect()


Model: minilm


Batches: 100%|██████████| 1246/1246 [03:14<00:00,  6.40it/s]


Shape: (9967, 384)
Vreme: 194.66 s
Brzina: 51.20 filmova/s
Sačuvano: artifacts\embeddings\minilm_embeddings.npy

Model: mpnet


Batches: 100%|██████████| 1246/1246 [32:57<00:00,  1.59s/it]   


Shape: (9967, 768)
Vreme: 1977.15 s
Brzina: 5.04 filmova/s
Sačuvano: artifacts\embeddings\mpnet_embeddings.npy

Model: gte_modernbert


Batches:  11%|█         | 138/1246 [29:12<3:54:28, 12.70s/it]


KeyboardInterrupt: 

In [21]:
model_benchmark = pd.DataFrame(benchmark_results)

display(model_benchmark)


,model,number_of_movies,embedding_dimension,encode_time_seconds,movies_per_second,embedding_file_mb
0,minilm,9967,384,194.655145,51.203373,14.600220
1,mpnet,9967,768,1977.153789,5.041085,29.200317


T4 GPU na colabu je ovo uradio za 20 sekundi.

In [18]:
from pathlib import Path
import numpy as np

embedding_path = Path("artifacts/embeddings/gte_modernbert_embeddings.npy")

print(embedding_path.exists())

gte_embeddings = np.load(embedding_path)

print(gte_embeddings.shape)
print(gte_embeddings.dtype)

assert len(df) == len(gte_embeddings)

print("Broj embeddinga odgovara broju filmova.")

True
(9967, 768)
float16
Broj embeddinga odgovara broju filmova.


In [19]:
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = Path("data/processed/movies_cleaned.csv")
EMBEDDINGS_DIR = Path("artifacts/embeddings")

EMBEDDING_PATHS = {
    "minilm": EMBEDDINGS_DIR / "minilm_embeddings.npy",
    "mpnet": EMBEDDINGS_DIR / "mpnet_embeddings.npy",
    "gte_modernbert": EMBEDDINGS_DIR / "gte_modernbert_embeddings.npy",
}


# Load dataset
movies_df = pd.read_csv(DATA_PATH)

# Load embedding matrices
embeddings = {
    model_name: np.load(file_path)
    for model_name, file_path in EMBEDDING_PATHS.items()
}

print(f"Number of movies: {len(movies_df):,}")
print(f"Dataset shape: {movies_df.shape}")
print()


validation_rows = []

for model_name, matrix in embeddings.items():
    if matrix.ndim != 2:
        raise ValueError(
            f"{model_name}: expected a 2D matrix, got shape {matrix.shape}"
        )

    row_count_matches = matrix.shape[0] == len(movies_df)
    nan_count = int(np.isnan(matrix).sum())
    inf_count = int(np.isinf(matrix).sum())

    l2_norms = np.linalg.norm(matrix, axis=1)

    validation_rows.append(
        {
            "model": model_name,
            "shape": str(matrix.shape),
            "num_movies": matrix.shape[0],
            "embedding_dimension": matrix.shape[1],
            "dtype": str(matrix.dtype),
            "row_count_matches": row_count_matches,
            "nan_count": nan_count,
            "inf_count": inf_count,
            "mean_l2_norm": l2_norms.mean(),
            "std_l2_norm": l2_norms.std(),
            "min_l2_norm": l2_norms.min(),
            "max_l2_norm": l2_norms.max(),
        }
    )

    if not row_count_matches:
        raise ValueError(
            f"{model_name}: {matrix.shape[0]} embeddings, "
            f"but dataset contains {len(movies_df)} movies."
        )

    if nan_count > 0:
        raise ValueError(f"{model_name}: found {nan_count} NaN values.")

    if inf_count > 0:
        raise ValueError(f"{model_name}: found {inf_count} infinite values.")


embedding_validation_df = pd.DataFrame(validation_rows)

embedding_validation_df

Number of movies: 9,967
Dataset shape: (9967, 12)



,model,shape,num_movies,embedding_dimension,dtype,row_count_matches,nan_count,inf_count,mean_l2_norm,std_l2_norm,min_l2_norm,max_l2_norm
0,minilm,"(9967, 384)",9967,384,float32,True,0,0,1.0,4.135931e-08,1.000000,1.0
1,mpnet,"(9967, 768)",9967,768,float32,True,0,0,1.0,4.590942e-08,1.000000,1.0
2,gte_modernbert,"(9967, 768)",9967,768,float16,True,0,0,1.0,2.441406e-04,0.999512,1.0
